<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab04.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 4 — Hamiltonians and Energy Estimation from Shots

**Maps to:** Module 3, Lessons 5–6 (Hamiltonian; Energy Estimation)

**Time:** ~60 minutes (instructor walkthrough ~15 min)

---

### The question this lab answers

A quantum computer can only tell you `0` or `1` on each wire. Module 3 says the energy is

$$ E = \langle\psi|H|\psi\rangle,\qquad
H = c_0 I + c_1 Z_0 + c_2 Z_1 + c_3 Z_0Z_1 + c_4 (X_0X_1 + Y_0Y_1) .$$

So: how do you get a *number in Hartree* out of a *pile of bitstrings*? And how many
bitstrings do you need? That second question is why the module ends with the slide
"1,000,000,000 shots."

### After this lab you can
1. Turn measurement counts into an expectation value $\langle P\rangle$ for any Pauli string.
2. Reproduce the lecture's numerical example line by line.
3. Measure $\langle X_0X_1\rangle$ and $\langle Y_0Y_1\rangle$ using $H$ and $S^\dagger H$
   basis rotations, on a machine that only measures $Z$.
4. Group commuting terms so 5 Pauli terms need only **3** measurement settings.
5. Predict the shot cost of reaching chemical accuracy, and of a 50-term Hamiltonian.

In [ ]:
# %pip install -q qiskit qiskit-aer matplotlib
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

np.set_printoptions(precision=4, suppress=True)
sim = AerSimulator()
HARTREE_TO_EV = 27.2114
print("ready")

## Part A — From counts to energy, by hand

### A.1 The rule

Measure every qubit in the $Z$ basis. Each shot gives a bitstring. For a Pauli string
made of $Z$'s and $I$'s, assign

$$ \text{bit } 0 \rightarrow +1, \qquad \text{bit } 1 \rightarrow -1 ,$$

multiply the values of the qubits the operator acts on, and average over shots.
Equivalently: **the sign is $+1$ if the number of 1s on the relevant qubits is even,
$-1$ if it is odd.** That is the parity rule from Lab 2, now read backwards.

> ### ⚠️ Bit order, again
> The lecture writes bitstrings as $q_0 q_1$ (left to right). **Qiskit prints them
> reversed**: `'01'` from Qiskit means $q_0=1,\ q_1=0$. In Part A we use the lecture's
> ordering because we are copying its numbers. In Part B we reverse Qiskit's strings
> first (`bits[::-1]`) so the two agree. Get this wrong and your energy comes out with
> $c_1$ and $c_2$ swapped — a very common bug.

### Exercise 1 — implement the estimator

In [ ]:
def expval_from_counts(counts, qubits, lecture_order=True):
    '''Expectation value of a Z-type Pauli string from measurement counts.'''
    total = sum(counts.values())
    acc = 0
    for bits, n in counts.items():
        b = bits if lecture_order else bits[::-1]
        # TODO: count how many of the relevant qubits are 1;
        #       add +n if that count is even, -n if it is odd
        ...
    return acc / total

demo = {"00": 0, "01": 810, "10": 190, "11": 0}
print("<Z0>   =", expval_from_counts(demo, [0]))        # expect  0.62
print("<Z1>   =", expval_from_counts(demo, [1]))        # expect -0.62
print("<Z0Z1> =", expval_from_counts(demo, [0, 1]))     # expect -1.0
assert np.isclose(expval_from_counts(demo, [0]), 0.62)
assert np.isclose(expval_from_counts(demo, [1]), -0.62)
assert np.isclose(expval_from_counts(demo, [0, 1]), -1.0)
print("PASS -- matches the lecture slide.")

### A.2 The $X$ and $Y$ settings

$X_0X_1$ and $Y_0Y_1$ are not diagonal, so you cannot read them off a $Z$ measurement.
Use the Lab 3 identity backwards:

* $HXH = Z$ → apply $H$ to both qubits, then measure $Z$; the parity you read is
  $\langle X_0X_1\rangle$.
* $H S^\dagger Y S H = Z$ → apply $S^\dagger$ then $H$ to both qubits, then measure $Z$;
  the parity you read is $\langle Y_0Y_1\rangle$.

Once rotated, the arithmetic is identical — same parity rule, different circuit prefix.

### Exercise 2 — reproduce the lecture's energy

Lecture counts (illustrative), 1000 shots each:

| setting | counts |
|---|---|
| $Z$ | `00:0, 01:810, 10:190, 11:0` |
| $X$ (after $H,H$) | `00:400, 01:100, 10:100, 11:400` |
| $Y$ (after $S^\dagger H$) | `00:300, 01:200, 10:200, 11:300` |

In [ ]:
c0, c1, c2, c3, c4 = (-1.0523732458, +0.3979374248, -0.3979374248,
                      -0.0112801043, +0.1809311998)

counts_Z = {"00": 0,   "01": 810, "10": 190, "11": 0}
counts_X = {"00": 400, "01": 100, "10": 100, "11": 400}
counts_Y = {"00": 300, "01": 200, "10": 200, "11": 300}

Z0   = ...   # TODO
Z1   = ...   # TODO
Z0Z1 = ...   # TODO
X0X1 = ...   # TODO  (which counts dict?)
Y0Y1 = ...   # TODO

print(f"<Z0>={Z0:.2f}  <Z1>={Z1:.2f}  <Z0Z1>={Z0Z1:.2f}  "
      f"<X0X1>={X0X1:.2f}  <Y0Y1>={Y0Y1:.2f}")

E = ...      # TODO: assemble the energy from the coefficients
print(f"\nE = {E:.4f} Ha = {E*HARTREE_TO_EV:.2f} eV")
assert np.isclose(E, -0.4029, atol=1e-3)
print("PASS -- matches the lecture arithmetic.")

### A.3 Two conventions you must keep straight

**(i) Nuclear repulsion.** The qubit Hamiltonian above is the *electronic* part. The
proton–proton repulsion $E_\text{nuc} = 1/R$ is a plain number, not an operator, so it is
added classically at the end:

$$ E_\text{total} = \langle H_\text{qubit}\rangle + E_\text{nuc},
\qquad E_\text{nuc}(0.735\,\text{Å}) = 0.7200\ \text{Ha}. $$

The famous $-30.9$ eV for H$_2$ is a **total** energy: $-1.1373$ Ha $\times\ 27.2114$.

**(ii) $XX$ vs $XX+YY$.** In the two-qubit reduced model, $X_0X_1$ alone and
$\tfrac12(X_0X_1+Y_0Y_1)$ produce the *same* coupling between $|01\rangle$ and
$|10\rangle$ — the physical sector. So the two ways of writing the Hamiltonian agree only
if the $XX+YY$ version carries **half** the coefficient. Qiskit Nature (Lab 5) hands you
the $XX$-only form with $c_4 = 0.18093$; the equivalent $XX+YY$ form uses
$c_4/2 = 0.09047$ on each. Check it once here and you will never wonder again.

In [ ]:
Hxx     = SparsePauliOp.from_list([("II", c0), ("IZ", c1), ("ZI", c2),
                                   ("ZZ", c3), ("XX", c4)])
Hxx_yy  = SparsePauliOp.from_list([("II", c0), ("IZ", c1), ("ZI", c2),
                                   ("ZZ", c3), ("XX", c4/2), ("YY", c4/2)])
E_NUC = 0.7199689944489797

for name, Hop in [("XX only        ", Hxx), ("(XX+YY)/2 form ", Hxx_yy)]:
    e = np.linalg.eigvalsh(Hop.to_matrix())[0]
    print(f"{name} ground state: {e:+.6f} Ha (electronic)   "
          f"{e + E_NUC:+.6f} Ha (total) = {(e+E_NUC)*HARTREE_TO_EV:.2f} eV")

assert np.isclose(np.linalg.eigvalsh(Hxx.to_matrix())[0],
                  np.linalg.eigvalsh(Hxx_yy.to_matrix())[0])
print("\nPASS -- identical ground state, as promised. And -1.1373 Ha = -30.9 eV,")
print("the number on the Module 3 slide.")

## Part B — Measuring a real state

Now stop using given numbers. Build the H$_2$ ansatz circuit (the one from your
"Reduced Qubit Mapping" slide), measure it in three settings, and rebuild the energy.

The reference configuration (Hartree–Fock) is $|01\rangle$, prepared with a single `X`
gate; the CNOT–R$_z$–CNOT core with `S`/`H` wrappers is the Givens mixer from Lab 3.

In [ ]:
def h2_ansatz(theta):
    '''Reduced 2-qubit H2 ansatz: HF reference |01> plus a one-parameter mixer.'''
    qc = QuantumCircuit(2)
    qc.x(0)                                  # Hartree-Fock reference
    qc.s(1); qc.h(1); qc.h(0)                # basis / phase conventions
    qc.cx(0, 1); qc.rz(2*theta, 1); qc.cx(0, 1)
    qc.h(1); qc.sdg(1); qc.h(0)
    return qc

print(h2_ansatz(0.112).draw(output="text"))
sv = Statevector(h2_ansatz(0.112))
print("\nstate:", sv.probabilities_dict())
print("(Qiskit prints q1q0, so '01' here means q0=1 -- the HF configuration.)")

### Exercise 3 — three measurement settings

Complete `measure_setting`: copy the circuit, apply the basis rotation, measure all, run.

In [ ]:
def measure_setting(qc, basis, shots=8000, seed=11):
    '''Run qc with a basis rotation ('Z', 'X' or 'Y') and return counts.'''
    c = qc.copy()
    if basis == "X":
        ...          # TODO: H on both qubits
    elif basis == "Y":
        ...          # TODO: S-dagger then H on both qubits
    c.measure_all()
    return sim.run(transpile(c, sim), shots=shots, seed_simulator=seed).result().get_counts()

def measured_energy(theta, shots=8000, seed=11):
    qc = h2_ansatz(theta)
    cz = measure_setting(qc, "Z", shots, seed)
    cx = measure_setting(qc, "X", shots, seed+1)
    cy = measure_setting(qc, "Y", shots, seed+2)
    ev = lambda counts, qs: expval_from_counts(counts, qs, lecture_order=False)
    vals = {"Z0": ev(cz, [0]), "Z1": ev(cz, [1]), "Z0Z1": ev(cz, [0, 1]),
            "X0X1": ev(cx, [0, 1]), "Y0Y1": ev(cy, [0, 1])}
    E = ...     # TODO: assemble, remembering the c4/2 convention for the XX+YY form
    return E, vals

E_meas, vals = measured_energy(0.112)
print({k: round(v, 4) for k, v in vals.items()})
print(f"\nmeasured  E_total = {E_meas + E_NUC:+.5f} Ha")
E_exact = np.real(Statevector(h2_ansatz(0.112)).expectation_value(Hxx_yy))
print(f"exact     E_total = {E_exact + E_NUC:+.5f} Ha")
assert abs(E_meas - E_exact) < 0.02
print("\nPASS")

### Exercise 4 — a first look at the energy landscape

Scan $\theta$ and plot the measured energy against the exact curve. This is the same
landscape the optimizer will walk downhill in Lab 5 — you are just doing it by brute
force first.

In [ ]:
import matplotlib.pyplot as plt

thetas = np.linspace(-0.6, 0.6, 25)
E_shot = [measured_energy(t, shots=2000, seed=3)[0] + E_NUC for t in thetas]
E_true = [np.real(Statevector(h2_ansatz(t)).expectation_value(Hxx_yy)) + E_NUC
          for t in thetas]

i = int(np.argmin(E_true))
print(f"exact minimum at theta = {thetas[i]:.3f}, E = {E_true[i]:.5f} Ha "
      f"= {E_true[i]*HARTREE_TO_EV:.2f} eV")

plt.figure(figsize=(6, 3.6))
plt.plot(thetas, E_true, "k-", label="exact  $\\langle\\psi(\\theta)|H|\\psi(\\theta)\\rangle$")
plt.plot(thetas, E_shot, "o", ms=4, label="2000 shots / setting")
plt.axhline(-1.1373, color="r", ls="--", lw=1, label="true ground state")
plt.xlabel(r"$\theta$"); plt.ylabel("total energy (Ha)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

Notice how shallow the valley is: the whole well is about 0.02 Ha deep, while a
2000-shot estimate wobbles by several mHa. **The optimizer has to find a minimum that is
barely deeper than the noise.** That single picture explains most of what makes VQE hard.

## Part C — Counting the cost

### C.1 How many settings, really?

The lecture asks: "if we run 1000 shots per axis, does that mean 3000 shots?"
The answer is yes — but note *why* it is 3 and not 5. $Z_0$, $Z_1$, and $Z_0Z_1$ all
commute qubit-wise, so **one** $Z$-basis experiment gives all three. Qiskit can find
these groups for you.

In [ ]:
groups = Hxx_yy.paulis.group_qubit_wise_commuting()
print(f"{len(Hxx_yy)} Pauli terms  ->  {len(groups)} measurement settings")
for g in groups:
    print("   ", [p.to_label() for p in g])

### Exercise 5 — shot noise scaling and chemical accuracy

*Chemical accuracy* is 1 kcal/mol $\approx 1.6$ mHa. Estimate how many shots per setting
you need to get there.

In [ ]:
shot_list = [250, 1000, 4000, 16000, 64000]
E_ref = np.real(Statevector(h2_ansatz(0.112)).expectation_value(Hxx_yy))

print(f"{'shots':>7} {'mean |err| (mHa)':>18}")
errs = []
for n in shot_list:
    # TODO: average |E_measured - E_ref| over ~8 different seeds at this shot count
    e = ...
    errs.append(np.mean(e))
    print(f"{n:7d} {1000*np.mean(e):18.3f}")

A = np.mean([e * np.sqrt(n) for e, n in zip(errs, shot_list)])
n_needed = (A / 0.0016)**2
print(f"\nfitted  error ~ {A:.4f} / sqrt(N)")
print(f"shots per setting for 1.6 mHa: ~{n_needed:,.0f}")

### Exercise 6 — the "VQE is expensive" slide, as a calculation

Reproduce the arithmetic at the end of Module 3 and then improve it.

In [ ]:
def shot_budget(n_terms, shots_per_term, n_energy_evals, n_geometry_steps):
    per_energy = n_terms * shots_per_term
    per_inner  = per_energy * n_energy_evals
    total      = per_inner * n_geometry_steps
    return per_energy, per_inner, total

pe, pi, tot = shot_budget(50, 1000, 200, 100)
print(f"worst case (no grouping): {pe:,} / energy   {pi:,} / inner loop   {tot:,} total")

# With qubit-wise grouping a 50-term Hamiltonian typically collapses to ~10-15 settings.
pe2, pi2, tot2 = shot_budget(12, 1000, 200, 100)
print(f"with grouping (12 groups): {pe2:,} / energy   {pi2:,} / inner loop   {tot2:,} total")
print(f"\nspeedup from grouping alone: {tot/tot2:.1f}x")

seconds_per_shot = 1e-4
print(f"\nAt {seconds_per_shot*1e6:.0f} us/shot the ungrouped run takes "
      f"{tot*seconds_per_shot/3600:.1f} hours of pure QPU time.")
print("This is why term grouping, shot allocation, and cheap ansatze are")
print("research topics, not implementation details.")

## Checkpoint

1. You measure `{'00': 250, '01': 250, '10': 250, '11': 250}` in the $Z$ setting.
   What are $\langle Z_0\rangle$ and $\langle Z_0Z_1\rangle$?
2. Why can $\langle Z_0\rangle$ and $\langle Z_0Z_1\rangle$ share one experiment, but
   $\langle Z_0Z_1\rangle$ and $\langle X_0X_1\rangle$ cannot?
3. You quadruple the shots. By what factor does the statistical error fall?
4. The identity term $c_0 I$ has the largest coefficient in the H$_2$ Hamiltonian. How
   many shots does it need? Why?
5. If a 4-qubit Hamiltonian has 15 Pauli terms but only 5 commuting groups, what is the
   saving in QPU time, and what has *not* improved?

### What is next
**Lab 5 (capstone)** puts Labs 1–4 together: molecule → Hamiltonian → ansatz → inner
optimization loop → outer bond-length loop → the H$_2$ dissociation curve.